In [30]:
import scipy.io as sio
from sklearn.decomposition import PCA
from torch_geometric.data import InMemoryDataset, Data
import torch
import torch.nn as nn
import scipy.sparse as sp
import numpy as np


class CitationDomainData(InMemoryDataset):
    """
    引文数据集加载器：acmv9,citationv1,dblpv7 originally from CDNE
    """
    def __init__(self,root,name,use_pca=True,pca_dim=1000,transform=None,pre_transform=None,pre_filter=None):
        self.name=name
        self.use_pca = use_pca
        self.dim = pca_dim
        super(CitationDomainData, self).__init__(root, transform, pre_transform, pre_filter)
        self.data, self.slices = torch.load(self.processed_paths[0])
    @property
    def raw_file_names(self):
        return [f'{self.name}.mat']
    @property
    def processed_file_names(self):
        if self.use_pca:
            return [f'data_pca_{self.dim}.pt']
        else:
            return ['data.pt']

    def feature_compression(self,features):
        """Preprcessing of features"""
        features = features.toarray()
        feat = sp.lil_matrix(PCA(n_components=self.dim, random_state=0).fit_transform(features))
        return feat.toarray()

    def process(self):
        net = sio.loadmat(self.raw_dir+"\\"+self.name+".mat")
        features, adj, labels = net['attrb'], net['network'], net['group']
        if not isinstance(features, sp.lil_matrix):
            features = sp.lil_matrix(features)
        # citation networks do not use PCA, but blog networks use
        if self.use_pca:
            features = self.feature_compression(features)
            features = torch.from_numpy(features).to(torch.float)
        else:
            features = torch.from_numpy(features.todense()).to(torch.float)
        if not isinstance(adj,sp.coo_matrix):
            adj = sp.coo_matrix(adj)
        indices = np.vstack([adj.row, adj.col])
        edge_index = torch.tensor(indices,dtype=torch.long)
        # label is float: to support BCEWithLogits loss
        y = torch.from_numpy(np.array(labels)).to(torch.float)
        data_list = []
        graph = Data(x=features,edge_index=edge_index,y=y)
        # train-val-test split
        random_node_indices = np.random.permutation(y.shape[0])
        train_size = int(len(random_node_indices) * 0.7)
        val_size = int(len(random_node_indices) * 0.1)
        train_node_indices = random_node_indices[:train_size]
        val_node_indices = random_node_indices[train_size:train_size+val_size]
        test_node_indices = random_node_indices[train_size+val_size:]
        train_mask = torch.zeros([y.shape[0]], dtype=torch.uint8)
        train_mask[train_node_indices] = 1
        val_mask = torch.zeros([y.shape[0]], dtype=torch.uint8)
        val_mask[val_node_indices] = 1
        test_mask = torch.zeros([y.shape[0]], dtype=torch.uint8)
        test_mask[test_node_indices] = 1
        graph.train_mask = train_mask
        graph.val_mask = val_mask
        graph.test_mask = test_mask
        if self.pre_transform is not None:
            graph = self.pre_transform(graph)
        data_list.append(graph)
        data, slices = self.collate(data_list)
        torch.save((data, slices), self.processed_paths[0])

In [39]:
from modelUtils import *
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')
import copy


class GCNClassifier(nn.Module):
    def __init__(self,input_dim,output_dim):
        self.input_dim = input_dim
        self.output_dim = output_dim
        super(GCNClassifier, self).__init__()
        self.h1 = 512
        self.h2 = 128
        self.conv1 = GCNConv(self.input_dim, self.h1)
        self.conv2 = GCNConv(self.h1, self.h2)
        self.fc = nn.Linear(self.h2, self.output_dim)
        self.prelu = nn.PReLU()
    def forward(self,data):
        x, edge_index = data.x, data.edge_index
        feat1 = F.dropout(self.prelu(self.conv1(x, edge_index)))
        feat2 = F.dropout(self.prelu(self.conv2(feat1, edge_index)))
        logits = self.fc(feat2)
        return logits

def f1_scores(y_pred, y_true):
    """ y_pred: prob  y_true 0/1 """
    def predict(y_tru, y_pre):
        top_k_list = y_tru.sum(1)
        prediction = []
        for i in range(y_tru.shape[0]):
            pred_i = torch.zeros(y_tru.shape[1])
            pred_i[torch.argsort(y_pre[i, :])[-int(top_k_list[i].item()):]] = 1
            prediction.append(torch.reshape(pred_i, (1, -1)))
        prediction = torch.cat(prediction, dim=0)
        return prediction.to(torch.int32)
    results = {}
    predictions = predict(y_true, y_pred)
    averages = ["micro", "macro"]
    for average in averages:
        results[average] = f1_score(y_true.cpu().numpy(), predictions.cpu().numpy(), average=average)
    return results

def test(inp,net):
    net.eval()
    with torch.no_grad():
        test_out = net(inp)[inp.test_mask]
    pred = F.sigmoid(test_out)
    result = f1_scores(pred,inp.y[inp.test_mask])
    return result

def test_tgt(inp, net):
    net.eval()
    with torch.no_grad():
        test_out = net(inp)
    pred = F.sigmoid(test_out)
    result = f1_scores(pred,inp.y)
    return result

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
datasets = ["acmv9","citationv1","dblpv7"]
for i in range(len(datasets)):
    for j in range(i+1,len(datasets)):
        dual_ds = [(i,j),(j,i)]
        for ds in dual_ds:
            src_name = datasets[ds[0]]
            tgt_name = datasets[ds[1]]
            print(src_name, tgt_name)
            src = CitationDomainData(f"data/{src_name}",name=src_name,use_pca=False)
            tgt = CitationDomainData(f"data/{tgt_name}",name=tgt_name,use_pca=False)
            src_data = src[0].to(device)
            tgt_data = tgt[0].to(device)
            inp_dim = src_data.x.shape[1]
            out_dim = src_data.y.shape[1]
            model = GCNClassifier(inp_dim, out_dim).to(device)
            criterion = nn.BCEWithLogitsLoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
            model.zero_grad()
            model.train()
            running_loss = 0.
            best_macro = 0.
            best_micro = 0.
            best_gcn_wts = copy.deepcopy(model.state_dict())
            for epoch in range(200):
                optimizer.zero_grad()
                out = model(src_data)
                loss = criterion(out[src_data.train_mask], src_data.y[src_data.train_mask])
                running_loss += loss
                loss.backward()
                optimizer.step()
                if epoch%10==0 and epoch>0:
                    test_res = test(src_data, model)
                    tgt_res = test_tgt(tgt_data, model)
                    print(f"EPOCH:{epoch}, Loss:{running_loss/10}, Source: MacroF1={test_res['macro']}, MicroF1={test_res['micro']}, Target: MacroF1={tgt_res['macro']}, MicroF1={tgt_res['micro']}")
                    if tgt_res['macro']>best_macro:
                        best_macro = tgt_res['macro']
                        best_gcn_wts = copy.deepcopy(model.state_dict())
                    if tgt_res['micro']>best_micro:
                        best_micro = tgt_res['micro']
                    running_loss = 0.
            torch.save(best_gcn_wts, f"checkpoint/{src_name}-{tgt_name}-gcn.pt")
            with open('gcn-results.txt','a+',encoding='utf8') as f:
                f.write(src_name+'-'+tgt_name+','+'macro '+str(best_macro)+','+'micro '+str(best_micro)+"\n")

cuda
acmv9 citationv1
EPOCH:10, Loss:0.5007262825965881, Source: MacroF1=0.7820890407040816, MicroF1=0.8012048192771084, Target: MacroF1=0.5992240472657373, MicroF1=0.6836611361388858
EPOCH:20, Loss:0.16434001922607422, Source: MacroF1=0.7953329949849236, MicroF1=0.7896586345381527, Target: MacroF1=0.6004191342239532, MicroF1=0.6447643116141083
EPOCH:30, Loss:0.12058261781930923, Source: MacroF1=0.8269399323961274, MicroF1=0.8283132530120482, Target: MacroF1=0.7291611471232964, MicroF1=0.7517855180749367
EPOCH:40, Loss:0.09712667018175125, Source: MacroF1=0.8385317680260345, MicroF1=0.8433734939759037, Target: MacroF1=0.7111448736691707, MicroF1=0.7307988133172177
EPOCH:50, Loss:0.08655737340450287, Source: MacroF1=0.8270219796540713, MicroF1=0.8288152610441767, Target: MacroF1=0.7166313101185591, MicroF1=0.7362927150862543
EPOCH:60, Loss:0.08557433634996414, Source: MacroF1=0.8478947800030386, MicroF1=0.8493975903614458, Target: MacroF1=0.7203110301461593, MicroF1=0.7394791781122954
E

Processing...
Done!


EPOCH:10, Loss:0.44684693217277527, Source: MacroF1=0.8151344576455806, MicroF1=0.8247991967871486, Target: MacroF1=0.6021053955223012, MicroF1=0.673940716493908
EPOCH:20, Loss:0.16143380105495453, Source: MacroF1=0.8554006430120538, MicroF1=0.856425702811245, Target: MacroF1=0.6778654240253666, MicroF1=0.7055828332424077
EPOCH:30, Loss:0.10850562900304794, Source: MacroF1=0.8480467781269505, MicroF1=0.8468875502008032, Target: MacroF1=0.6500931670641599, MicroF1=0.6843062374977269
EPOCH:40, Loss:0.15544553101062775, Source: MacroF1=0.6104294259382163, MicroF1=0.6270080321285141, Target: MacroF1=0.42927047753754677, MicroF1=0.5393707946899436
EPOCH:50, Loss:0.15833993256092072, Source: MacroF1=0.8493765462747855, MicroF1=0.8473895582329318, Target: MacroF1=0.6834543202630959, MicroF1=0.7114020731042008
EPOCH:60, Loss:0.09247108548879623, Source: MacroF1=0.846732480027212, MicroF1=0.8468875502008032, Target: MacroF1=0.6579739620730662, MicroF1=0.6950354609929078
EPOCH:70, Loss:0.0835684